# Filter AMR papers with LLM

Use the `llm.py` helper to ask a model whether a paper's abstract asserts that a pathogen is showing **significant resistance** to a given antibiotic.


In [1]:

# --- LLM filtering configuration ---
import pandas as pd

from llm import create_llm_client
from amr.filtering import (
    DEFAULT_SYSTEM_PROMPT,
    prepare_pmids_to_review,
    run_llm_filter,
)

SYSTEM_PROMPT = """
You are an expert microbiologist reviewing evidence. Given a title and abstract, decide whether the authors explicitly state or conclude that the specified pathogen is showing clinically meaningful resistance to the specified antibiotic.

Rules:
- Answer "yes" only if the abstract clearly reports resistance, reduced susceptibility, treatment failure due to resistance, or emergence of resistant isolates for that pathogen/antibiotic pair.
- Answer "no" if the abstract does not make that claim, is about a different organism/drug, only mentions testing without resistance, or only provides background.
- Answer "uncertain" if the abstract is missing, ambiguous, or only indirectly related.
- Base the decision only on the provided text; do not infer beyond it.

Return compact JSON with keys: decision (yes/no/uncertain) and rationale (one short sentence).
"""

LLM_PROVIDER = "openai"
MODEL_NAME = "gpt-4o-mini"
NCBI_EMAIL = "your_email@example.com"
NEW_PMIDS_TO_PROCESS = 100000  # number of new PMIDs to process in this run (None for all)
N_PMIDS_PER_RUN = 100  # how many unprocessed PMIDs to handle per run
RATE_LIMIT = 0.35  # seconds between PubMed calls
OUTPUT_PATH = "pmids_results_filtered.parquet"
LOG_PATH = "pmid_filter.log"
MAX_LLM_CALLS = None  # set to an integer to cap spend per run
DRY_RUN = False  # set True to skip LLM calls
SAVE_EVERY = 50  # checkpoint frequency

llm_kwargs = {}
if MODEL_NAME:
    llm_kwargs["model"] = MODEL_NAME

llm = create_llm_client(provider=LLM_PROVIDER, **llm_kwargs)


In [2]:

import os
import pandas as pd
from amr.filtering import prepare_pmids_to_review

# Load previous decisions
if os.path.exists(OUTPUT_PATH):
    reviewed_df = pd.read_parquet(OUTPUT_PATH)
    reviewed = set(reviewed_df["pmid"].astype(str)) if not reviewed_df.empty else set()
    print(f"Resuming with {len(reviewed)} PMIDs already reviewed from {OUTPUT_PATH}")
else:
    reviewed_df = None
    reviewed = set()

initial_reviewed_count = len(reviewed)

pmid_df = pd.read_parquet("pmids_results.parquet")
# Build full exploded table to report totals
full_long_df = prepare_pmids_to_review(pmid_df, exclude_pmids=None)
full_total = len(full_long_df)

# Exclude reviewed first, THEN cap to the next chunk
long_df = full_long_df
if reviewed:
    long_df = long_df[~long_df["pmid"].astype(str).isin(reviewed)]

pmid_total = len(long_df)
reviewed_count = len(reviewed)
print(f"Total PMIDs in source (exploded): {full_total}")
print(f"Loaded {pmid_total} candidate PMIDs after exclusion; reviewed set size: {reviewed_count}")
if NEW_PMIDS_TO_PROCESS is None:
    print("Planning to process all remaining PMIDs this run.")
else:
    print(f"Planning to process up to {NEW_PMIDS_TO_PROCESS} new PMIDs this run.")


Resuming with 75470 PMIDs already reviewed from pmids_results_filtered.parquet
Total PMIDs in source (exploded): 518057
Loaded 93582 candidate PMIDs after exclusion; reviewed set size: 75470
Planning to process up to 100000 new PMIDs this run.


In [ ]:

%%time

iter_idx = 0

while True:
    remaining_df = long_df[~long_df["pmid"].astype(str).isin(reviewed)]
    processed_this_run = len(reviewed) - initial_reviewed_count

    if NEW_PMIDS_TO_PROCESS is not None:
        remaining_quota = NEW_PMIDS_TO_PROCESS - processed_this_run
        if remaining_quota <= 0:
            print(f"Reached NEW_PMIDS_TO_PROCESS={NEW_PMIDS_TO_PROCESS}; stopping.")
            break
        remaining_df = remaining_df.head(remaining_quota)

    if remaining_df.empty:
        print("No PMIDs left to process.")
        break

    batch_df = remaining_df.head(N_PMIDS_PER_RUN)
    remaining_after_batch = max(len(remaining_df) - len(batch_df), 0)
    iter_idx += 1
    print(
        f"Iteration {iter_idx}: processing {len(batch_df)} PMIDs; {remaining_after_batch} still queued"
    )

    filtered_df = run_llm_filter(
        batch_df,
        llm,
        email=NCBI_EMAIL,
        rate_limit=RATE_LIMIT,
        save_every=SAVE_EVERY,
        system_prompt=SYSTEM_PROMPT,
        progress_every=10,
        resume_from=reviewed_df,
        seen_pmids=reviewed,
        output_path=OUTPUT_PATH,
        log_path=LOG_PATH,
        max_llm_calls=MAX_LLM_CALLS,
        dry_run=DRY_RUN,
    )

    reviewed_df = filtered_df
    reviewed = set(reviewed_df["pmid"].astype(str))

    # Build positive-only dataset and save
    pmid_df_full = pd.read_parquet("pmids_results.parquet")

    # Merge decisions onto exploded PMIDs
    positive_flags = filtered_df[filtered_df["decision"].str.lower() == "yes"]["pmid"].copy()
    positive_rows = (
        pmid_df_full.explode("pmids")
        .assign(pmid=lambda d: d["pmids"].astype(str).str.extract(r"(\d+)")[0])
        .dropna(subset=["pmid"])
        .merge(positive_flags, on="pmid", how="inner")
    )

    # Reassemble to original grain
    pmids_positive = (
        positive_rows.groupby(["pathogen", "antibiotic", "start_date", "end_date"], as_index=False)
        .agg(
            pmids=("pmid", list),
            count=("pmid", "size"),
            venue_histogram=("venue_histogram", "first"),
        )
    )

    positive_path = "pmids_results_positive.parquet"
    pmids_positive.to_parquet(positive_path, index=False)
    print(f"Saved positive-only dataset to {positive_path} with {len(pmids_positive)} rows")


Iteration 1: processing 100 PMIDs; 93482 still queued
Processed 10/100     no:5    yes:5    uncertain:0   |   cumulative 75480     no:46913    yes:25430    uncertain:3137
Processed 20/100     no:10    yes:10    uncertain:0   |   cumulative 75490     no:46918    yes:25435    uncertain:3137
Processed 30/100     no:14    yes:14    uncertain:2   |   cumulative 75500     no:46922    yes:25439    uncertain:3139
Processed 40/100     no:16    yes:21    uncertain:3   |   cumulative 75510     no:46924    yes:25446    uncertain:3140
Processed 60/100     no:28    yes:26    uncertain:6   |   cumulative 75530     no:46936    yes:25451    uncertain:3143
Processed 70/100     no:34    yes:27    uncertain:9   |   cumulative 75540     no:46942    yes:25452    uncertain:3146
Processed 80/100     no:38    yes:32    uncertain:10   |   cumulative 75550     no:46946    yes:25457    uncertain:3147
Saved positive-only dataset to pmids_results_positive.parquet with 21576 rows
Iteration 2: processing 100 PMIDs; 9